## LR Interaction

In [ ]:
import os
import sys
import argparse
import pandas as pd
import torch
import anndata as ad
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from DeepRUOT.losses import OT_loss1
from DeepRUOT.utils import (
    generate_steps, load_and_merge_config,
    SchrodingerBridgeConditionalFlowMatcher,
    generate_state_trajectory, get_batch, get_batch_size
)
from DeepRUOT.train import train_un1_reduce, train_all
from DeepRUOT.models import FNet_interaction, scoreNet2
from DeepRUOT.constants import DATA_DIR, RES_DIR
from DeepRUOT.exp import setup_exp

### Load config

In [ ]:
config_path = '../config/mosta_config.yaml'

# Load and merge configuration
config = load_and_merge_config(config_path)

### Load data and model

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]
#df = df[df.iloc[:,1] > 0.4]
device = torch.device('cpu')
exp_dir, logger = setup_exp(
            RES_DIR, 
            config, 
            config['exp']['name']
        )
dim = config['data']['dim']

In [ ]:
model_config = config['model']
        
f_net = FNet_interaction(
            in_out_dim=model_config['in_out_dim'],
            hidden_dim=model_config['hidden_dim'],
            n_hiddens=model_config['n_hiddens'],
            activation=model_config['activation'],
            use_spatial = True, 
            num_heads = 8,
            thre = 0.06,
            num_layers = 1,

        ).to(device)

sf2m_score_model = scoreNet2(
    in_out_dim=model_config['in_out_dim'],
    hidden_dim=model_config['score_hidden_dim'],
    activation=model_config['activation']
).float().to(device)

In [ ]:
f_net.load_state_dict(torch.load(os.path.join(exp_dir, 'model_final'),map_location=torch.device('cpu')))
f_net.to(device)
sf2m_score_model.load_state_dict(torch.load(os.path.join(exp_dir, 'score_model'),map_location=torch.device('cpu')))
sf2m_score_model.to(device)

### Save Attention

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
device = 'cpu'
f_net = f_net.to(device)
time_points = df['samples'].unique()


In [ ]:

data_by_time = {}

for time in time_points:
    subset = df[df['samples'] == time]
    data = subset.iloc[:,1:].values
    lnw0 = torch.log(torch.ones(subset.shape[0],1) / (subset.shape[0])).to(device)
    data = torch.tensor(data, dtype=torch.float32).to(device)
    data.requires_grad_(True)
    time_tensor = torch.tensor(time, dtype=torch.float32).to(device)
    # with torch.no_grad():
        # net_forces= f_net.interaction_net(data, lnw0, time_tensor, return_attn=True)
    # attn = f_net.interaction_net.gnn_layers[0].attn
    # print(attn)
    # attn = torch.abs(attn)
    attn_mean = np.load('attn_mean_t0.npy')
    # attn_mean = attn.mean(dim = 1).cpu().numpy() #.mean(dim = 1)
    # edge_index = f_net.interaction_net.edge_index
    # edge_index = edge_index.cpu().numpy()
    # edge_index = edge_index[:, edge_index[0] != edge_index[1]]
    edge_index = np.load('edge_index_t0.npy')
    attn_matrix = np.zeros((data.shape[0], data.shape[0]))
    attn_matrix[edge_index[0], edge_index[1]] = attn_mean
    #store attn_matrix
    # 带上时间点做文件名（按需修改）
    out_path = f"attn_matrix_t{time}.npy"
    # np.save(out_path, attn_matrix)
    print(attn_matrix.shape)
    spatial_coord = data[:, :2].detach().cpu().numpy()
    # Find top 100 strongest interactions
    # Get indices of top 100 values excluding self-interactions
    top_k = 10000
    indices = np.where(attn_matrix > 0)  # Get non-zero indices
    values = attn_matrix[indices]
    
    # Filter indices based on spatial coordinates
    # mask = ((spatial_coord[indices[1], 0] >= 0.6) & 
    #         (spatial_coord[indices[1], 0] <= 0.85) &
    #         (spatial_coord[indices[1], 1] <= 0.2))
    mask = np.ones(len(indices[0]), dtype=bool)
    
    filtered_indices = (indices[0][mask], indices[1][mask])
    filtered_values = values[mask]
    
    # Sort by attention values and get top k
    top_indices = np.argsort(filtered_values)[-top_k:]
    
    # Get source and target coordinates for filtered interactions
    source_coords = spatial_coord[filtered_indices[0][top_indices]]
    target_coords = spatial_coord[filtered_indices[1][top_indices]]
    
    # Plot spatial coordinates and interactions
    plt.figure(figsize=(8, 12))
    
    # Plot all cells
    plt.scatter(spatial_coord[:, 0], 
               spatial_coord[:, 1], 
               c='lightgray', alpha=0.5)
    
    # Plot filtered interactions as arrows
    # Get attention scores for the filtered interactions
    attention_scores = filtered_values[top_indices]
    # Normalize attention scores to use for arrow sizes
    norm_scores = (attention_scores - attention_scores.min()) / (attention_scores.max() - attention_scores.min())
    
    # Calculate vector field from interactions
    x_grid = np.linspace(spatial_coord[:, 0].min(), spatial_coord[:, 0].max(), 50)
    y_grid = np.linspace(spatial_coord[:, 1].min(), spatial_coord[:, 1].max(), 50)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    U = np.zeros_like(X)
    V = np.zeros_like(Y)
    W = np.zeros_like(X)  # For vector magnitude/weights
    
    # Accumulate vectors at each grid point
    for i in range(len(top_indices)):
        dx = target_coords[i, 0] - source_coords[i, 0]
        dy = target_coords[i, 1] - source_coords[i, 1]
        
        # Find nearest grid points to source coordinate
        x_idx = np.abs(x_grid - source_coords[i, 0]).argmin()
        y_idx = np.abs(y_grid - source_coords[i, 1]).argmin()
        
        # Add weighted vectors
        U[y_idx, x_idx] += dx * norm_scores[i]
        V[y_idx, x_idx] += dy * norm_scores[i]
        W[y_idx, x_idx] += norm_scores[i]
    
    # Normalize vectors
    mask = W > 0
    U[mask] /= W[mask]
    V[mask] /= W[mask]
    
    # Plot streamlines
    plt.streamplot(X, Y, U, V, density=5, color='blue', 
                  linewidth=W/W.max()*2, arrowsize=1.5)
    
    plt.title(f'Top {top_k} Strongest Interactions')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()
    # # Plot attention matrix heatmap
    # plt.figure(figsize=(10, 8))
    # sns.heatmap(attn_matrix, cmap='YlOrRd', cbar_kws={'label': 'Attention Weight'})
    # plt.title(f'Attention Matrix at Time {time}')
    # plt.xlabel('Target Cell')
    # plt.ylabel('Source Cell')
    # plt.show()

    # Plot attention heatmap using seaborn
    # plt.figure(figsize=(8, 6))
    # attn_mean = attn.mean(dim=0).cpu().numpy()
    # sns.heatmap(attn_mean, cmap='YlOrRd', cbar_kws={'label': 'Attention Weight'})
    # plt.title(f'Attention Map at Time {time}')
    # plt.xlabel('Target Cell')
    # plt.ylabel('Source Cell')
    # plt.show()

### load adata

In [ ]:
import scanpy as sc

# 加载 h5ad 文件
adata = sc.read("../spatial_data/Mouse_embryo_all_stage.h5ad")

# 查看数据的基本信息
print(adata)

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance

# 假设 adata 是您的原始 AnnData 对象
# 获取所有唯一的批次名称
batch_names = adata.obs['timepoint'].cat.categories

# 创建字典存储每个批次的 AnnData 对象，确保是实际对象
adata_dict = {}
for batch in batch_names:
    adata_dict[batch] = adata[adata.obs['timepoint'] == batch].copy()

# 定义预处理函数
def preprocess_adata(adata):
    #sc.pp.normalize_total(adata, target_sum=1e4)
    #sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata = adata[:, adata.var.highly_variable]
    return adata

# 定义空间坐标缩放函数
def scale_spatial_coords(adata):
    spatial_coords = adata.obsm['spatial']
    x_min, x_max = spatial_coords[:, 0].min(), spatial_coords[:, 0].max()
    y_min, y_max = spatial_coords[:, 1].min(), spatial_coords[:, 1].max()
    x_range = x_max - x_min
    spatial_coords[:, 0] = (spatial_coords[:, 0] - x_min) / x_range
    scale_factor = 1 / x_range
    spatial_coords[:, 1] = (spatial_coords[:, 1] - y_min) * scale_factor
    adata.obsm['spatial'] = spatial_coords # 更新修改后的坐标
    return adata

# 定义绘图函数
def plot_spatial(adata, batch_name):
    spatial_coords = adata.obsm['spatial']
    annotations = adata.obs['annotation']
    annotation_colors = adata.uns['annotation_colors']
    category_to_color = dict(zip(annotations.cat.categories, annotation_colors))
    colors = annotations.map(category_to_color)
    plt.figure(figsize=(10, 8))
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=colors, s=10, alpha=1)
    plt.gca().invert_yaxis()  # 反转y轴
    plt.title(f'Spatial Visualization for {batch_name}')
    plt.xlabel('Scaled X')
    plt.ylabel('Scaled Y')
    plt.show()


def remove_outliers(adata, radius=0.05, threshold=5):
    # 获取空间坐标
    spatial_coords = adata.obsm['spatial']
    # 计算所有数据点之间的距离矩阵
    dist_matrix = distance.cdist(spatial_coords, spatial_coords)
    # 计算每个点的邻居数量（减去自身）
    neighbors = np.sum(dist_matrix < radius, axis=1) - 1
    # 创建掩码：保留邻居数量大于等于阈值的点
    mask = neighbors >= threshold
    # 应用掩码，删除离群点
    adata = adata[mask]
    return adata

# 自动化处理每个批次
for batch in batch_names:
    adata_batch = adata_dict[batch].copy()
    # adata_batch = preprocess_adata(adata_batch)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    # adata_batch = remove_outliers(adata_batch, radius=0.05, threshold=5)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    adata_dict[batch]=adata_batch.copy()

import numpy as np
import pandas as pd

# 初始化标签列表
labels_list = []
T = 5
batch_indices = [3, 4, 5, 6]

for t, batch_idx in enumerate(batch_indices):
    adata_t = adata_dict[batch_names[batch_idx]]
    labels_t = adata_t.obs['annotation'].values  # 获取当前时间点的 Annotation
    labels_list.append(labels_t)

# 合并所有标签
all_labels = np.concatenate(labels_list)

# 读取 CSV 文件
df_new = pd.read_csv('../data/mosta_four_time.csv')

# 添加 Annotation 列
df_new['Annotation'] = all_labels
# 获取 Annotation 的类别和颜色
if pd.api.types.is_categorical_dtype(adata.obs['annotation']):
    categories = adata.obs['annotation'].cat.categories
else:
    categories = adata.obs['annotation'].unique()  # 如果不是 categorical 类型

colors = adata.uns['annotation_colors']

# 创建标签到颜色的映射
label_to_color = dict(zip(categories, colors))

### LR Interaction

In [ ]:
# LR投影和空间可视化
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scanpy as sc
from matplotlib.patches import FancyArrowPatch
# 修复 numpy Inf 问题
if not hasattr(np, 'Inf'):
    np.Inf = np.inf
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
import commot as ct
import plotly.graph_objects as go
import pickle
import os
from matplotlib.patches import FancyArrowPatch

In [ ]:
# 加载LR数据库
def load_lr_database():
    """加载配体-受体数据库"""
    # 这里使用COMMOT内置的数据库，您也可以加载自定义数据库
    lr_df = pd.read_csv("/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/database/CellChatDB.ligrec.mouse.csv")  # 替换为您的LR数据库路径
    return lr_df

In [ ]:
print(adata_dict[batch_names[3]].obs.columns)

In [ ]:
# 执行分析
# 加载LR数据库
lr_db = load_lr_database()

# 获取当前分析的细胞类型
cell_types = adata_dict[batch_names[3]].obs['annotation'].unique()

In [ ]:
print(lr_db.columns)
print(lr_db.head())

In [ ]:
# 投影到LR空间
def project_to_lr_space(adata, edge_index, attn_weights, lr_db, cell_types):
    """
    将注意力权重投影到配体-受体空间
    """
    # 创建通信矩阵
    comm_matrix = np.zeros((len(cell_types), len(cell_types)))
    
    # 获取细胞类型映射
    cell_type_to_idx = {ct: i for i, ct in enumerate(cell_types)}
    
    # 聚合注意力到细胞类型级别
    for i, (src, tgt) in enumerate(edge_index.T):
        src_type = adata.obs['annotation'].iloc[src]
        tgt_type = adata.obs['annotation'].iloc[tgt]
        
        if src_type in cell_type_to_idx and tgt_type in cell_type_to_idx:
            src_idx = cell_type_to_idx[src_type]
            tgt_idx = cell_type_to_idx[tgt_type]
            comm_matrix[src_idx, tgt_idx] += attn_weights[i]
            
    # 细胞层面归一化
    comm_matrix = comm_matrix / comm_matrix.max()
    
    # ========== 2. 基因表达严谨还原 ==========
    raw_expr = adata.X
    if hasattr(raw_expr, "toarray"):
        raw_expr = raw_expr.toarray()

    # Log(1+x) -> Count Space
    expr_matrix_counts = np.expm1(raw_expr)

    labels = adata.obs['annotation'].values

    # ========== 3. 计算 Cell Type Mean Expression (Count 空间) ==========
    type_mean_expr = []
    for ct in cell_types:
        mask = (labels == ct)
        if mask.sum() > 0:
            type_mean_expr.append(expr_matrix_counts[mask, :].mean(axis=0))
        else:
            type_mean_expr.append(np.zeros(expr_matrix_counts.shape[1]))
    type_mean_expr = np.array(type_mean_expr)

    gene_to_idx = {g: idx for idx, g in enumerate(adata.var_names)}

    # ========== 4. 计算 LR Scores ==========
    lr_scores = {}

    for _, lr_pair in lr_db.iterrows():
        ligand, receptor = lr_pair['ligand'], lr_pair['receptor']
        if ligand not in gene_to_idx or receptor not in gene_to_idx:
            continue

        lig_idx = gene_to_idx[ligand]
        rec_idx = gene_to_idx[receptor]

        lr_score = np.zeros((len(cell_types), len(cell_types)))

        for i in range(len(cell_types)):
            for j in range(len(cell_types)):
                ligand_mean = type_mean_expr[i, lig_idx]
                receptor_mean = type_mean_expr[j, rec_idx]
                comm_strength = comm_matrix[i, j]

                # 严格物理意义：L * R * Communication
                lr_score[i, j] = ligand_mean * receptor_mean * comm_strength

        lr_scores[f"{ligand}_{receptor}"] = lr_score

    # ========== 5. 保存 ==========
    np.save('lr_scores_t{}.npy', lr_scores)
    np.save('comm_matrix_t{}.npy', comm_matrix)
    
    return lr_scores, comm_matrix

In [ ]:
def load_lr_scores(time_key, results_dir="lr_projection_results"):
    
    file_path = os.path.join(results_dir, f"lr_scores_{time_key}.pkl")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"LR scores file not found: {file_path}")
    
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    
    print(f"Loaded LR scores for {time_key}")
    print(f"  - Number of LR pairs: {len(data['lr_scores'])}")
    print(f"  - Cell types: {data['cell_types']}")
    
    return data


In [ ]:
with open("all_timepoint_communications_merged.pkl", "rb") as f:
    all_time_communications = pickle.load(f)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches

In [ ]:
adata_dict

In [ ]:
# === 遍历时间点执行投影 ===
output_dir = "lr_projection_results"
os.makedirs(output_dir, exist_ok=True)
# 确定使用的时间点(从您的代码中提取)
batch_indices = [3, 4, 5, 6]  # 对应数组索引
time_keys = [batch_names[i] for i in batch_indices]  # ['E13.5', 'E14.5', 'E15.5', 'E16.5']
print(f"分析时间点: {time_keys}")

for time_key, comm_data in all_time_communications.items():
    print({time_key})
    adata = adata_dict[time_key]
    
    # 载入 edge_index 与 attn 数据（假设你命名一致）
    edge_index_file = f"edge_index_t{time_keys.index(time_key)}.npy"
    attn_matrix_file = f"attn_matrix_t{time_keys.index(time_key)}.npy"
    
    edge_index = np.load(edge_index_file)
    attn_matrix = np.load(attn_matrix_file)
    attn_weights = attn_matrix
    print(attn_weights.shape)
    # 获取细胞类型列表
    cell_types = comm_data['types']
    
    # 投影到 LR 空间
    lr_scores, comm_matrix = project_to_lr_space(
        adata=adata,
        edge_index=edge_index,
        attn_weights=attn_weights,
        lr_db=lr_db,
        cell_types=cell_types
    )
    
    # 保存结果
    save_path = os.path.join(output_dir, f"lr_scores_{time_key}.pkl")
    with open(save_path, 'wb') as f:
        pickle.dump({
            'time_key': time_key,
            'lr_scores': lr_scores,
            'comm_matrix': comm_matrix,
            'cell_types': cell_types
        }, f)
    
    print(f"{time_key} 投影完成，保存到 {save_path}")


In [ ]:
# === 遍历时间点执行投影 ===
output_dir = "lr_projection_results"
os.makedirs(output_dir, exist_ok=True)
# 确定使用的时间点(从您的代码中提取)
time_keys = ['E11.5']  # 插值数据
t_values = ['-1'] 
print(f"分析时间点: {time_keys}")

for time_key, comm_data in all_time_communications.items():
    if time_key in time_keys:
        print({time_key})
        # 找到time_key在time_keys中的索引
        idx = time_keys.index(time_key)
        # 获取对应的t值
        t = float(t_values[idx])
        t_str = f"{t:.3f}".replace('.', 'p').replace('-', 'n')
        
        adata = sc.read(f"/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t{t_str}_with_genes.h5ad")
        
        # 载入 edge_index 与 attn 数据（假设你命名一致）
        edge_index_file = f"edge_index_interp_t{t_str}.npy"
        attn_matrix_file = f"attn_mean_interp_t{t_str}.npy"
        
        edge_index = np.load(edge_index_file)
        attn_matrix = np.load(attn_matrix_file)
        attn_weights = attn_matrix
        print(attn_weights.shape)
        
        # 获取细胞类型列表
        cell_types = comm_data['types']
        
        # 投影到 LR 空间
        lr_scores, comm_matrix = project_to_lr_space(
            adata=adata,
            edge_index=edge_index,
            attn_weights=attn_weights,
            lr_db=lr_db,
            cell_types=cell_types
        )
        
        # 保存结果
        save_path = os.path.join(output_dir, f"lr_scores_{time_key}.pkl")
        with open(save_path, 'wb') as f:
            pickle.dump({
                'time_key': time_key,
                'lr_scores': lr_scores,
                'comm_matrix': comm_matrix,
                'cell_types': cell_types
            }, f)
        
        print(f"{time_key} 投影完成，保存到 {save_path}")


In [ ]:
allowed_stages = ["E12.5", "E13.5", "E14.5", "E15.5"]
print(adata.obs.columns)   

In [ ]:
adata_filtered = adata[adata.obs["timepoint"].isin(allowed_stages)].copy()

print(adata_filtered)

In [ ]:
#adata_filtered.write("adata_4stage.h5ad")

In [ ]:
adata_filtered=sc.read("adata_4stage.h5ad")
print(adata_filtered)

In [ ]:
adata_n1=sc.read("/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_tn1p000_with_genes.h5ad")

In [ ]:
import anndata as ad

adata_new = ad.concat(
    [adata_n1, adata_filtered[adata_filtered.obs["timepoint"].isin(["E12.5"])].copy(), adata_filtered[adata_filtered.obs["timepoint"].isin(["E13.5"])].copy(), adata_filtered[adata_filtered.obs["timepoint"].isin(["E14.5"])].copy(), adata_filtered[adata_filtered.obs["timepoint"].isin(["E15.5"])].copy()],
    join="outer",      # outer = 基因全集对齐（最安全）
    axis=0,            # 按细胞合并
    label="batch",     # 在 obs 里新增一列 'batch'
    keys=["E11.5","E12.5", "E13.5", "E14.5", "E15.5"],  # 两个 batch 名字
    fill_value=0       # 缺失的基因补 0
)

In [ ]:
adata_new

### LR Interaction with Imputation

In [ ]:
adata_new.obs["batch"]

In [ ]:

# 获取所有唯一的批次
batches = adata_new.obs['batch'].unique()
print(f"总共有 {len(batches)} 个批次")

# 逐个查看每个批次的空间坐标
for batch in batches:
    # 获取当前批次的索引
    batch_indices = adata_new.obs['batch'] == batch
    
    # 获取当前批次的空间坐标
    batch_spatial = adata_new.obsm["spatial"][batch_indices]
    
    print(f"\n批次 '{batch}' 的信息:")
    print(f"  细胞/点数量: {len(batch_spatial)}")
    print(f"  空间坐标形状: {batch_spatial.shape}")
    print(f"  坐标范围: X: {batch_spatial[:, 0].min():.2f} - {batch_spatial[:, 0].max():.2f}")
    print(f"            Y: {batch_spatial[:, 1].min():.2f} - {batch_spatial[:, 1].max():.2f}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def commot_spatial_communication_field_v3(
        adata,
        lr_pair,
        time_list,
        results_dir="lr_projection_results",
        spatial_key="spatial"
    ):
    """
    接口不变，绘图逻辑严格对齐 plot_temporal_spatial_trajectory:
    - Ligand / Receptor: 原始 log 表达
    - Interaction Potential: Count(L)*Count(R)
    - Hotspot: Step2 LR score 映射回细胞并归一化
    """
    ligand = lr_pair.split("_")[0]
    receptor = "_".join(lr_pair.split("_")[1:])
    
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    # 1. 提取目标时间点
    adata_sub = adata[adata.obs['timepoint'].isin(time_list)].copy()

    # 预设列
    for col in ["ligand_expr", "receptor_expr", "lr_interaction_potential", "lr_comm_hotspot"]:
        adata_sub.obs[col] = 0.0

    # 2. 逐时间点计算
    for t_key in time_list:
        print(f"Calculating {lr_pair} scores for {t_key}...")
        mask = adata_sub.obs['timepoint'] == t_key
        loc_idx = np.where(mask)[0]
            
        try:
            data = load_lr_scores(t_key, results_dir)
            lr_scores, cell_types = data["lr_scores"], data["cell_types"]
        except Exception as e:
            print(f"Skip {t_key} loading: {e}")
            continue

        if lr_pair not in lr_scores:
            continue

        # --- A. L / R 表达 (Count 空间) ---
        l_expr_log = adata_sub[mask, ligand].X
        r_expr_log = adata_sub[mask, receptor].X
        if hasattr(l_expr_log, "toarray"): l_expr_log = l_expr_log.toarray().flatten()
        if hasattr(r_expr_log, "toarray"): r_expr_log = r_expr_log.toarray().flatten()
        
        # Log -> Count
        l_count = np.expm1(l_expr_log)
        r_count = np.expm1(r_expr_log)
        
        adata_sub.obs.iloc[loc_idx, adata_sub.obs.columns.get_loc("ligand_expr")] = l_count
        adata_sub.obs.iloc[loc_idx, adata_sub.obs.columns.get_loc("receptor_expr")] = r_count
        
        # --- B. Interaction Potential ---
        pot_count = l_count * r_count
        adata_sub.obs.iloc[loc_idx, adata_sub.obs.columns.get_loc("lr_interaction_potential")] = pot_count

        # --- C. Hotspot: Step2 Score -> 映射回细胞 ---
        lr_matrix = lr_scores[lr_pair]  # Count 空间
        ct_to_idx = {ct: i for i, ct in enumerate(cell_types)}
        annos = adata_sub.obs.loc[mask, "annotation"].values

        cell_scores = np.zeros(len(annos))
        for i, ct in enumerate(cell_types):
            m = (annos == ct)
            if m.sum() > 0:
                # Receiver 端总强度
                cell_scores[m] = lr_matrix[:, i].sum()

        # min-max 归一化
        if cell_scores.max() > cell_scores.min():
            cell_scores = (cell_scores - cell_scores.min()) / (cell_scores.max() - cell_scores.min())
        else:
            cell_scores[:] = 0

        adata_sub.obs.iloc[loc_idx, adata_sub.obs.columns.get_loc("lr_comm_hotspot")] = cell_scores

    # 3. 绘图配置（名字不变）
    plot_configs = [
        ("ligand_expr", "Reds", f"Ligand_{ligand}", f"Ligand: {ligand}"),
        ("receptor_expr", "Blues", f"Receptor_{receptor}", f"Receptor: {receptor}"),
        ("lr_comm_hotspot", "viridis", "Comm_Hotspot", "Communication Hotspots"),
        ("lr_interaction_potential", "plasma", "Interaction_Potential", "Interaction Potential")
    ]

    # 4. 执行绘图
    for obs_key, cmap, file_suffix, title in plot_configs:
        n = len(time_list)
        fig, axes = plt.subplots(1, n, figsize=(6*n, 8), facecolor="white")
        if n == 1:
            axes = [axes]

        sc_last = None  # 用来接 colorbar

        for i, t_key in enumerate(time_list):
            ax = axes[i]
            mask = adata_sub.obs['timepoint'] == t_key

            sc = ax.scatter(
                adata_sub.obsm[spatial_key][mask, 0],
                adata_sub.obsm[spatial_key][mask, 1],
                c=adata_sub.obs.loc[mask, obs_key],
                cmap=cmap,
                s=5,
                alpha=0.85,
                edgecolors='none'
            )
            sc_last = sc

            # 标题只放时间点
            ax.set_title(str(t_key), fontsize=18, pad=10)

            # 完全去掉坐标轴、边框、刻度
            ax.set_axis_off()
            for spine in ax.spines.values():
                spine.set_visible(False)

            # 只有后几个切片 invert
            if i != 0:
                ax.invert_yaxis()

        # 总标题
        fig.suptitle(title, fontsize=24, fontweight="bold", y=0.98)

        # 公用 colorbar
        cb = fig.colorbar(
            sc_last,
            ax=axes,
            fraction=0.02,
            pad=0.02
        )
        cb.outline.set_visible(False)
        cb.ax.tick_params(labelsize=14)

        save_path = os.path.join(results_dir, f"{lr_pair}_AllTimes_{file_suffix}.pdf")
        plt.savefig(save_path, dpi=300, bbox_inches="tight", transparent=False)
        plt.close(fig)

        print(f"Successfully saved: {save_path}")



In [ ]:
commot_spatial_communication_field_v3(
        adata=adata_new,
        lr_pair="H2-Q4_Cd8b1",
        time_list=['E11.5','E12.5', 'E13.5', 'E14.5', 'E15.5'],
        results_dir="lr_projection_results",
        spatial_key="spatial"
    )

In [ ]:
commot_spatial_communication_field_v3(
        adata=adata_new,
        lr_pair="Ptprm_Ptprm",
        time_list=['E11.5', 'E12.5', 'E13.5', 'E14.5', 'E15.5'],
        results_dir="lr_projection_results",
        spatial_key="spatial"
    )

In [ ]:
commot_spatial_communication_field_v3(
        adata=adata_new,
        lr_pair="Scgb3a2_Marco",
        time_list=['E11.5', 'E12.5', 'E13.5', 'E14.5', 'E15.5'],
        results_dir="lr_projection_results",
        spatial_key="spatial"
    )

In [ ]:
commot_spatial_communication_field_v3(
        adata=adata_new,
        lr_pair="H2-Q10_Cd8b1",
        time_list=['E11.5', 'E12.5', 'E13.5', 'E14.5', 'E15.5'],
        results_dir="lr_projection_results",
        spatial_key="spatial"
    )

In [ ]:
import json

# 读取ipynb文件
with open('mosta_LR_interaction.ipynb', 'r', encoding='utf-8') as f:
    notebook = json.load(f)

# 提取所有代码
all_code = []
for cell in notebook['cells']:
    if cell['cell_type'] == 'code':
        code = ''.join(cell['source'])
        all_code.append(code)

# 保存到.py文件
with open('mosta_interaction_code.py', 'w', encoding='utf-8') as f:
    for i, code in enumerate(all_code, 1):
        f.write(f'# Cell {i}\n')
        f.write(code)
        f.write('\n\n' + '#'*50 + '\n\n')

print("代码已保存到 mosta_interaction_code.py")